<center><h2><strong><font color="blue"> Data Engineering for Data Science - Week 06</font></strong></h2></center>

<center><img alt="" src="images/cover.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">DEDS-05: Advanced Selection, Aggregations, & Normalization</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

<center><h2><strong><font color="blue">Outline</font></strong></h2></center>

**Topics**:

* DDL (Create, Alter, Drop) vs. DML (Select, Insert, Update, Delete).
* Advanced Selection: Joins (Inner, Left, Right, Full), Aggregations, and Window Functions.
* The importance of data redundancy and anomalies in research data.
* Functional Dependencies.
* Normal Forms: 1NF, 2NF, 3NF, and BCNF.
* Denormalization: When and why to break the rules.

  
**Lab**: 
* Given a flat, messy "Excel-style" dataset, decompose it into a 3NF schema.


# Export-Import Data using phpMyAdmin

### Please clone (“pull”) the course GitHub repository and locate the file "example-schema-sample-data.sql" within the "data" folder.

<center><img alt="" src="images/deds-04/phpmyadmin-eximport.jpg" style="height: 480px;"/></center>

In [5]:
# Best Practice; use a function to connect to our database
import warnings; warnings.simplefilter('ignore')
import mysql.connector
import pandas as pd

host = "localhost"
user = "root"
password = ""
database = "university_registry_db"

def connect(host="localhost", user="root", password="", database=""):
    try:
        db = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database)
        con = db.cursor()
        print("Connected to the '{}' Database.".format(database))
        return db, con
    except Exception as err_:
        print("Connection to the Database server failed:")
        print(err_)

db, con = connect(host=host, user=user, password=password, database=database)

Connected to the 'university_registry_db' Database.


## Data Manipulation Language (DML) in MariaDB
While Data Definition Language (DDL) builds the "house," **Data Manipulation Language (DML)** is the act of moving furniture in. It allows researchers to interact with the actual data stored within the tables. In the context of the **CRUD** (Create, Read, Update, Delete) cycle, DML is where the majority of your daily data engineering work occurs.

## 1. The SELECT Statement (Reading Data)

The `SELECT` statement is the most frequent operation in data science. It retrieves specific records from one or more tables based on defined criteria.

### Basic Syntax
```sql
SELECT full_name, email 
FROM students 
WHERE country_origin = 'Indonesia' 
ORDER BY full_name ASC;
```

* **Pro Tip:** Avoid `SELECT *` in production code. Explicitly naming columns reduces memory overhead and prevents your Python scripts from breaking if the table schema changes later.

## 2. The INSERT Statement (Creating Data)

`INSERT` adds new rows to a table. For production-grade research, we often handle this through Python scripts to ensure data is sanitized.

### Standard Insertion
```sql
INSERT INTO instructors (full_name, email) 
VALUES ('Dr. Aris Kusuma', 'aris.k@uiii.ac.id');
```

### Multi-row Insertion
```sql
INSERT INTO students (full_name, country_origin) 
VALUES 
    ('Siti Aminah', 'Indonesia'),
    ('John Doe', 'Australia'),
    ('Li Wei', 'China');
```

## 3. The UPDATE Statement (Modifying Data)

The `UPDATE` statement modifies existing records. In a professional environment, this must be handled with extreme precision.

### Critical Usage
```sql
UPDATE enrollments 
SET grade = 3.85 
WHERE student_id = 101 AND course_id = 5;
```

> **Warning:** Never omit the `WHERE` clause unless you intended to change the value for every single row in the table. There is no "undo" button for a mass update in a basic configuration.

## 4. The DELETE Statement (Removing Data)

The `DELETE` statement removes specific rows from a table. Unlike the `DROP` command (which removes the table itself), `DELETE` only removes the data.

### Targeted Deletion
```sql
DELETE FROM students 
WHERE enrollment_date < '2020-01-01';
```
* **Comparison:** Use `TRUNCATE` if you want to wipe all data from a table while keeping the structure intact. `TRUNCATE` is faster than `DELETE FROM table_name` because it doesn't log individual row deletions.

## DML Command Lifecycle

| Command | CRUD Equivalent | Research Utility |
| :--- | :--- | :--- |
| **SELECT** | Read | Extracting features for ML models or EDA. |
| **INSERT** | Create | Ingesting sensor data or web-scraped results. |
| **UPDATE** | Update | Cleaning "dirty" data or updating model logs. |
| **DELETE** | Delete | Removing outliers or outdated experimental runs. |

## Technical Summary
* **Logical Integrity:** Always verify your `WHERE` clauses using a `SELECT` statement before executing an `UPDATE` or `DELETE`.
* **Atomicity:** For multiple related changes, use **Transactions** (`START TRANSACTION`, `COMMIT`) to ensure the "all or nothing" principle of ACID compliance is met.
* **Automation:** In our next steps, we will automate these commands using Python loops to handle thousands of records simultaneously.

# Module 4.13: MariaDB Security Fundamentals (XAMPP for Windows)
In the default XAMPP installation, MariaDB is configured for ease of use rather than security—meaning the `root` user has no password. For production-grade research or even shared local environments, you must implement a strict security protocol. This module covers the essential Data Control Language (DCL) commands and administrative procedures.

## 1. Accessing the MariaDB Console
On a Windows machine using XAMPP, you must first access the command-line interface. 
1. Open the **XAMPP Control Panel**.
2. Click the **Shell** button on the right-hand side.
3. In the terminal, type the following to log in as root (assuming no password yet):
   ```bash
   mysql -u root
   ```
## 2. Securing the Root Account
The first step in any deployment is securing the superuser account. In MariaDB, we use the `ALTER USER` command to set or change passwords.

### Setting a New Root Password
```sql
ALTER USER 'root'@'localhost' IDENTIFIED BY 'your_strong_password_here';
FLUSH PRIVILEGES;
```
* **Note:** After running this, your subsequent logins will require the `-p` flag: `mysql -u root -p`.
* **XAMPP Specific:** If you change the root password, you must also update the configuration in `phpMyAdmin` (found in `config.inc.php`) or you will lose access to the GUI.

## 3. User Account Management
Professional data engineering avoids using the `root` account for daily tasks. Instead, we create specific users for specific projects.

### Creating a New User
The syntax follows the pattern `'username'@'host'`. Using `'localhost'` ensures the user can only log in from the local machine.
```sql
CREATE USER 'researcher_dev'@'localhost' IDENTIFIED BY 'secure_pass_2026';
```

## 4. Granting Privileges and Roles
Privileges define what a user can actually do. We follow the **Principle of Least Privilege (PoLP)**: give users only the access they strictly need.

### a. Granting Specific Permissions
You can grant access at the global, database, or table level.
```sql
-- Grant full access to only one specific database
GRANT ALL PRIVILEGES ON university_registry_db.* TO 'researcher_dev'@'localhost';

-- Grant only "Read" access (SELECT)
GRANT SELECT ON university_registry_db.students TO 'researcher_dev'@'localhost';
```

### b. Using Roles
Roles are groups of privileges that can be assigned to multiple users, making management easier for large teams.
```sql
-- Create a role
CREATE ROLE 'data_analyst';

-- Assign privileges to the role
GRANT SELECT, SHOW VIEW ON *.* TO 'data_analyst';

-- Assign the role to a user
GRANT 'data_analyst' TO 'researcher_dev'@'localhost';
```

## 5. Reviewing and Revoking Access
Security is an ongoing process. You must periodically audit who has access to your research data.
* **View User Privileges:**
    ```sql
    SHOW GRANTS FOR 'researcher_dev'@'localhost';
    ```
* **Revoke Permissions:**
    ```sql
    REVOKE UPDATE, DELETE ON university_registry_db.* FROM 'researcher_dev'@'localhost';
    ```
* **Delete a User:**
    ```sql
    DROP USER 'researcher_dev'@'localhost';
    ```

<center><h2><strong><font color="blue">Understanding MariaDB Character Set and Collation</font></strong></h2></center>

In database management, these two terms are often grouped together, but they handle very different parts of how your data is processed. Think of it as the difference between **knowing which letters exist** and **knowing how to alphabetize them**.

## 1. Character Set (The "What")
A **Character Set** is a defined list of characters and their corresponding binary values. It determines **what** specific symbols, letters, and numbers the database is capable of storing.

* **Function:** It maps a character (like 'A' or '😊') to a specific number (like 65 or a 4-byte Unicode sequence) that the computer understands.
* **Common Example:** `utf8mb4`. This set is widely used because it can store almost every character from every language, plus emojis and mathematical symbols.

## 2. Collation (The "How")
A **Collation** is a set of rules used to **compare and sort** the characters within a specific character set. It determines the logic used in `ORDER BY` clauses or when checking equality in a `WHERE` clause.

* **Function:** It defines whether the database should be case-sensitive (is 'A' the same as 'a'?) and how to handle accents (is 'e' the same as 'é'?).
* **Common Example:** `utf8mb4_unicode_ci`. The `_ci` at the end stands for **Case Insensitive**, meaning the database treats 'Apple' and 'apple' as identical when searching or sorting.

## 3. Key Differences at a Glance

| Feature | Character Set | Collation |
| :--- | :--- | :--- |
| **Primary Goal** | Storage and encoding. | Sorting and comparison. |
| **Responsibility** | Defines *which* characters are valid. | Defines *how* valid characters relate to each other. |
| **Analogy** | The symbols in an alphabet. | The rules in a dictionary for sorting words. |
| **Scope** | Determines disk space (e.g., 1-byte vs 4-byte). | Determines query results (e.g., sorting order). |

## 4. Why the Distinction Matters
If you choose the wrong **Character Set**, you might end up with "broken" characters (often appearing as `?` or ``) because the database doesn't recognize the symbol you're trying to save. 

If you choose the wrong **Collation**, your data will be stored correctly, but your search results might be unexpected—for instance, a search for "resume" might fail to find "Résumé," or your alphabetical list might put lowercase 'z' before uppercase 'A'.

<center><h2><strong><font color="blue">Understanding MariaDB Character Set and Collation</font></strong></h2></center>

<center><img alt="" src="images/deds-04/DB-collations-character-sets.png" style="height: 600px;"/></center>

### Related Queries

```sql
CREATE DATABASE research_db 
CHARACTER SET utf8mb4 
COLLATE utf8mb4_unicode_ci;
-----------------------------------
CREATE TABLE international_students (
    id INT PRIMARY KEY,
    student_name VARCHAR(100)
) CHARACTER SET utf8mb4 COLLATE utf8mb4_bin;
-----------------------------------
SHOW TABLE STATUS LIKE 'students';
```

<center><h2><strong><font color="blue">The MariaDB my.ini Configuration File</font></strong></h2></center>

<center><img alt="" src="images/deds-04/mysql-my-ini-file.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">Prologue ~ Normalization</font></strong></h2></center>

<center><img alt="" src="images/deds-04/Various-dataStorage-solutions.png" style="height: 600px;"/></center>

# Practice

* Start designing database schema for your thesis project

## Next Week's Discussions

* Conceptual vs. Logical vs. Physical Data Models.
* Entity-Relationship Diagrams (ERD).
* OLTP (Transactional) vs. OLAP (Analytical) systems.
* Introduction to Dimensional Modeling (Star Schema vs. Snowflake Schema).


<center><h2><strong><font color="blue">End of Module</font></strong></h2></center>

<center><img alt="" src="images/meme-cartoon/meme-db-joins.jpg" style="height: 480px;"/></center>